# Lemmatisation of YAKE Keywords on SageMaker

Follows the same **`PySparkProcessor`** pattern as `key_yake_ner_sagemaker.ipynb`:

1. Write `lem_ana_script.py` — the PySpark + Spark-NLP script that runs *inside* the managed container
2. Resolve IAM role and reuse the staged Spark-NLP assembly JAR on S3
3. Submit a SageMaker Processing Job — SageMaker provisions the instance, mounts inputs, runs the script, copies output to S3, then **automatically tears down the instance**

| | Value |
|---|---|
| **Input parquet** | `s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana/` |
| **Output parquet** | `s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/lem_ana/` |
| **New columns** | `lemmatized_keywords_aspects`, `lemmatized_keywords_suggestions` |

**Pipeline per column:**

| Step | Component | Purpose |
|---|---|---|
| 1 | Array → string join | Flatten keyword array to text |
| 2 | `DocumentAssembler` | Wrap text |
| 3 | `Tokenizer` | Tokenise |
| 4 | `LemmatizerModel` (`lemma_antbnc`) | Lemmatise each token |
| 5 | Collect `result` | Back to array column |

## 1. Environment Setup

In [1]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [2]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp==5.5.3" \
    "boto3"

print("All packages installed.")

All packages installed.


## 2. SageMaker v2.x Session Setup

In [3]:
import boto3
import sagemaker
from sagemaker import get_execution_role

boto_session = boto3.Session()
sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region  = boto_session.region_name or sm_session.boto_region_name
account = boto3.client("sts").get_caller_identity()["Account"]

BUCKET    = "sagemaker-us-west-2-493644444178"
prefix    = "spark-nlp/key-ana"   # reuse the already-staged JAR from key-ana

INPUT_S3  = f"s3://{BUCKET}/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana"
OUTPUT_S3 = f"s3://{BUCKET}/spark-nlp/redaction/jgh-msss-prem/redacted_output/lem_ana"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Execution role ARN    : {role}")
print(f"Input  S3             : {INPUT_S3}")
print(f"Output S3             : {OUTPUT_S3}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml
SageMaker SDK version : 2.257.1
Region                : us-west-2
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd
Input  S3             : s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/key_ana
Output S3             : s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/lem_ana


## 3. Write PySpark Processing Script

This script runs **inside the SageMaker container** (`spark-submit`).
It reads the mounted input parquet, lemmatises the YAKE keyword array columns,
adds `lemmatized_keywords_aspects` and `lemmatized_keywords_suggestions`,
and writes the result to the output mount.

> All paths use `file://` — no HDFS, no S3 filesystem inside the container.

In [4]:
%%writefile lem_ana_script.py
"""
Spark-NLP lemmatisation processing script.
Executed inside a SageMaker PySparkProcessor container via spark-submit.

Reads the YAKE keyword columns produced by key_yake_ner_sagemaker.ipynb:
  yake_keywords_aspects      -> lemmatized_keywords_aspects
  yake_keywords_suggestions  -> lemmatized_keywords_suggestions

All intermediate I/O uses file:// (no HDFS).
"""
import os, sys, subprocess, glob, shutil, traceback

# ── Early diagnostics ─────────────────────────────────────────────────────────
print("=" * 60, flush=True)
print(f"Python  : {sys.version}", flush=True)
print(f"CWD     : {os.getcwd()}", flush=True)
print("=" * 60, flush=True)

# ── Install Python-side spark-nlp bindings (JAR loaded via --jars) ────────────
pip_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "spark-nlp==5.5.3", "pandas<2.0"],
    capture_output=True, text=True,
)
print(f"pip exit={pip_result.returncode}", flush=True)
if pip_result.returncode != 0:
    print("pip stderr:", pip_result.stderr[-3000:], flush=True)

try:
    import sparknlp
    from sparknlp.base import DocumentAssembler
    from sparknlp.annotator import Tokenizer, LemmatizerModel
    from pyspark.ml import Pipeline
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    print(f"Imports OK — spark-nlp {sparknlp.version()}", flush=True)
except Exception:
    print("IMPORT ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

INPUT_DIR  = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"

# ── Pretrained model cache → /tmp (writable inside container) ────────────────
CACHE_DIR = "/tmp/sparknlp_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Start Spark ───────────────────────────────────────────────────────────────
try:
    spark = (
        SparkSession.builder
        .appName("SparkNLP-LemAna")
        .config("spark.jsl.settings.pretrained.cache_folder", CACHE_DIR)
        .getOrCreate()
    )
    print(f"Spark {spark.version} started", flush=True)
except Exception:
    print("SPARKSESSION ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

# ── Read input parquet (file://) ─────────────────────────────────────────────
parquet_files = (
    glob.glob(os.path.join(INPUT_DIR, "**/*.parquet"), recursive=True)
    + glob.glob(os.path.join(INPUT_DIR, "*.parquet"))
)
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {INPUT_DIR}")

print(f"Reading parquet from: {INPUT_DIR}", flush=True)
spark_df = spark.read.parquet("file://" + INPUT_DIR)
print(f"Input  : {spark_df.count()} rows × {len(spark_df.columns)} columns", flush=True)
spark_df.printSchema()

# ── Columns to lemmatise (array<string> → array<string>) ─────────────────────
COLS = {
    "yake_keywords_aspects":     "lemmatized_keywords_aspects",
    "yake_keywords_suggestions": "lemmatized_keywords_suggestions",
}
missing = [c for c in COLS if c not in spark_df.columns]
if missing:
    raise ValueError(
        f"Expected columns {list(COLS)} not found. "
        f"Available: {spark_df.columns}"
    )
print(f"Will lemmatise columns: {list(COLS.keys())}", flush=True)

# ── Load lemmatiser once (shared across both columns) ─────────────────────────
print("Loading LemmatizerModel (lemma_antbnc, en) ...", flush=True)
lemmatizer = (
    LemmatizerModel.pretrained("lemma_antbnc", "en")
    .setInputCols(["token"])
    .setOutputCol("lemma")
)
print("LemmatizerModel loaded.", flush=True)

# ── Build pipeline ────────────────────────────────────────────────────────────
def build_lemma_pipeline(input_col: str) -> Pipeline:
    """
    Expects input_col to be a plain string column.
    Returns a Pipeline: DocumentAssembler -> Tokenizer -> LemmatizerModel.
    """
    doc = (
        DocumentAssembler()
        .setInputCol(input_col)
        .setOutputCol("document")
        .setCleanupMode("shrink")
    )
    tok = (
        Tokenizer()
        .setInputCols(["document"])
        .setOutputCol("token")
    )
    return Pipeline(stages=[doc, tok, lemmatizer])

# ── Process each column ───────────────────────────────────────────────────────
result_df = spark_df
for src_col, out_col in COLS.items():
    print(f"  Lemmatising: {src_col} -> {out_col}", flush=True)

    # Flatten array<string> to a single whitespace-joined string
    tmp_col = "_lem_input"
    working_df = result_df.withColumn(
        tmp_col,
        F.when(
            F.col(src_col).isNull() | (F.size(F.col(src_col)) == 0),
            F.lit("")
        ).otherwise(
            F.concat_ws(" ", F.col(src_col))
        )
    )

    pipeline = build_lemma_pipeline(tmp_col)
    # Fit on a tiny dummy frame so the pipeline is stateless
    dummy = spark.createDataFrame([[""]], [tmp_col])
    model = pipeline.fit(dummy)
    transformed = model.transform(working_df)

    # Collect lemmatised tokens back into an array, dedup
    result_df = (
        transformed
        .withColumn(out_col, F.array_distinct(F.col("lemma.result")))
        .drop("document", "token", "lemma", tmp_col)
    )
    print(f"    Done.", flush=True)

# ── Select original columns + new lemma columns in order ─────────────────────
output_cols = spark_df.columns + list(COLS.values())
output_df   = result_df.select(output_cols)

print(f"Output : {output_df.count()} rows, columns: {output_df.columns}", flush=True)

# ── Write to /tmp first, then copy to output mount (file://) ─────────────────
# Spark's overwrite mode deletes the target directory before writing.
# The SageMaker bind-mount blocks that delete, so we write to /tmp and
# copy the resulting parquet files manually.
TMP_PATH = "/tmp/spark_output"
if os.path.exists(TMP_PATH):
    shutil.rmtree(TMP_PATH)

output_df.coalesce(1).write.mode("overwrite").parquet("file://" + TMP_PATH)
print(f"Written to tmp: {TMP_PATH}", flush=True)

out_dir = os.path.join(OUTPUT_DIR, "data")
os.makedirs(out_dir, exist_ok=True)
for fname in os.listdir(TMP_PATH):
    src = os.path.join(TMP_PATH, fname)
    dst = os.path.join(out_dir, fname)
    shutil.copy2(src, dst)
    print(f"  Copied {fname} -> {dst}", flush=True)

print(f"Output ready at: {out_dir}", flush=True)

spark.stop()
print("Done.", flush=True)

Overwriting lem_ana_script.py


## 4. Resolve SageMaker-Assumable IAM Role

In [5]:
import boto3, json, time

iam          = boto3.client("iam")
SM_ROLE_NAME = "SageMakerExecutionRole-SparkNLP"

TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect"   : "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action"   : "sts:AssumeRole",
    }],
})

try:
    sm_role = iam.get_role(RoleName=SM_ROLE_NAME)["Role"]["Arn"]
    print(f"Reusing existing role: {sm_role}")

except iam.exceptions.NoSuchEntityException:
    print(f"Creating '{SM_ROLE_NAME}' ...")
    sm_role = iam.create_role(
        RoleName                 = SM_ROLE_NAME,
        AssumeRolePolicyDocument = TRUST_POLICY,
        Description              = "SageMaker execution role for Spark-NLP processing jobs",
    )["Role"]["Arn"]

    for policy_arn in [
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    ]:
        iam.attach_role_policy(RoleName=SM_ROLE_NAME, PolicyArn=policy_arn)
        print(f"  Attached {policy_arn}")

    print("Waiting 15 s for IAM propagation ...")
    time.sleep(15)

print(f"\nsm_role = {sm_role}")

Reusing existing role: arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP

sm_role = arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP


## 5. Verify / Re-Stage Spark-NLP Assembly JAR

The JAR was already uploaded by `key_yake_ner_sagemaker.ipynb`.
This cell checks it exists and re-uploads only if missing.

In [6]:
import urllib.request
from sagemaker.s3 import S3Uploader

NLP_VER         = "5.5.3"
jar_name        = f"spark-nlp-assembly-{NLP_VER}.jar"
sparknlp_jar_s3 = f"s3://{BUCKET}/{prefix}/jars/{jar_name}"

# Check if already staged
s3_client  = boto3.client("s3")
jar_key    = f"{prefix}/jars/{jar_name}"
try:
    s3_client.head_object(Bucket=BUCKET, Key=jar_key)
    print(f"JAR already staged: {sparknlp_jar_s3}")
except s3_client.exceptions.ClientError:
    print(f"JAR not found — downloading and staging ...")
    local = f"/tmp/{jar_name}"
    candidate_urls = [
        f"https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/jars/{jar_name}",
        (
            f"https://repo1.maven.org/maven2/com/johnsnowlabs/nlp/"
            f"spark-nlp-assembly_2.12/{NLP_VER}/"
            f"spark-nlp-assembly_2.12-{NLP_VER}.jar"
        ),
    ]
    for url in candidate_urls:
        try:
            print(f"  Trying {url}")
            urllib.request.urlretrieve(url, local)
            with open(local, "rb") as f:
                if f.read(2) != b"PK":
                    raise RuntimeError("Not a valid JAR")
            S3Uploader.upload(local, f"s3://{BUCKET}/{prefix}/jars", sagemaker_session=sm_session)
            print(f"  Staged at {sparknlp_jar_s3}")
            break
        except Exception as e:
            print(f"  ✗ {e}")
    else:
        raise RuntimeError("Could not obtain the Spark-NLP assembly JAR.")

print(f"\nsparknlp_jar_s3 = {sparknlp_jar_s3}")

JAR already staged: s3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar

sparknlp_jar_s3 = s3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar


## 6. Submit SageMaker Processing Job

`PySparkProcessor` provisions an `ml.m5.4xlarge` instance, runs `lem_ana_script.py`
inside the managed Spark container, copies output to S3, and **automatically
terminates the instance** when the job finishes (success or failure).

In [7]:
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

spark_processor = PySparkProcessor(
    base_job_name          = "spark-nlp-lem-ana",
    framework_version      = "3.3",
    role                   = sm_role,
    instance_count         = 1,
    instance_type          = "ml.m5.4xlarge",   # 16 vCPU / 64 GB
    max_runtime_in_seconds = 3600,
    sagemaker_session      = sm_session,
)

spark_processor.run(
    submit_app  = "lem_ana_script.py",
    submit_jars = [sparknlp_jar_s3],
    configuration = [{
        "Classification": "spark-defaults",
        "Properties": {
            "spark.serializer"                : "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max" : "2000M",
            "spark.driver.memory"             : "16G",
            "spark.driver.maxResultSize"      : "4000M",
            "spark.executor.memory"           : "20G",
            "spark.executor.memoryOverhead"   : "4G",
        },
    }],
    inputs = [
        ProcessingInput(
            source      = INPUT_S3,
            destination = "/opt/ml/processing/input",
        ),
    ],
    outputs = [
        ProcessingOutput(
            source      = "/opt/ml/processing/output/data",
            destination = OUTPUT_S3,
        )
    ],
    logs = True,
    wait = True,
)

print(f"\nJob complete. Results at: {OUTPUT_S3}")

INFO:sagemaker:Creating processing-job with name spark-nlp-lem-ana-2026-03-29-20-21-08-605


............03-29 20:23 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/lem_ana_script.py']
03-29 20:23 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-29 20:23 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/lem_ana_script.py']
03-29 20:23 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-29 20:23 smspark.cli  INFO     Initializing processing job.
03-29 20:23 smspark-submit INFO     {'current_host': 'algo-1', 'current_instance_type': 'ml.m5.4xlarge

## 7. Fetch CloudWatch Logs

In [8]:
import boto3, time

logs      = boto3.client("logs", region_name=region)
job_name  = spark_processor.latest_job.job_name
log_group = "/aws/sagemaker/ProcessingJobs"

print(f"Job      : {job_name}")
print(f"LogGroup : {log_group}\n")

for attempt in range(10):
    resp    = logs.describe_log_streams(logGroupName=log_group, logStreamNamePrefix=job_name)
    streams = resp.get("logStreams", [])
    if streams:
        break
    print(f"  Waiting for log streams (attempt {attempt+1}/10) ...")
    time.sleep(6)

for stream in streams:
    stream_name = stream["logStreamName"]
    print(f"\n{'='*60}\nStream: {stream_name}\n{'='*60}")
    events_resp = logs.get_log_events(
        logGroupName  = log_group,
        logStreamName = stream_name,
        startFromHead = True,
    )
    for event in events_resp["events"]:
        print(event["message"])

Job      : spark-nlp-lem-ana-2026-03-29-20-21-08-605
LogGroup : /aws/sagemaker/ProcessingJobs


Stream: spark-nlp-lem-ana-2026-03-29-20-21-08-605/algo-1-1774815707
03-29 20:23 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/lem_ana_script.py']
03-29 20:23 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-29 20:23 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/lem_ana_script.py']
03-29 20:23 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/key-ana/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-29 20:23 s

## 8. Preview Output from S3

In [9]:
import io, boto3, pandas as pd

s3_client  = boto3.client("s3")
out_prefix = OUTPUT_S3.replace(f"s3://{BUCKET}/", "").rstrip("/") + "/"

paginator    = boto3.client("s3").get_paginator("list_objects_v2")
output_files = [
    obj["Key"]
    for page in paginator.paginate(Bucket=BUCKET, Prefix=out_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]

print(f"Parquet files in output: {len(output_files)}")
for k in output_files:
    print(f"  s3://{BUCKET}/{k}")

if output_files:
    buf = io.BytesIO()
    s3_client.download_fileobj(BUCKET, output_files[0], buf)
    buf.seek(0)
    preview = pd.read_parquet(buf)
    print(f"\nPreview ({len(preview)} rows, columns: {list(preview.columns)}):")
    display(preview[["yake_keywords_aspects", "lemmatized_keywords_aspects",
                      "yake_keywords_suggestions", "lemmatized_keywords_suggestions"]].head(10))

Parquet files in output: 2
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/lem_ana/part-00000-44b95eef-4db3-4afa-b9d2-9488f5be7c96-c000.snappy.parquet
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/lem_ana/part-00000-4df57da9-3c5b-4f9e-9b73-6965eb92de63-c000.snappy.parquet

Preview (5740 rows, columns: ['Unnamed: 0', 'Date de complétion', 'Évaluation', "Aspects positifs de l'expérience globale", 'redacted_aspects', "Suggestions d'amélioration de l'expérience globale", 'redacted_suggestions', 'yake_keywords_aspects', 'yake_keywords_suggestions', 'lemmatized_keywords_aspects', 'lemmatized_keywords_suggestions']):


,yake_keywords_aspects,lemmatized_keywords_aspects,yake_keywords_suggestions,lemmatized_keywords_suggestions
0,"[doctor, thorough, review, told, us, done, tho...","[doctor, thorough, review, tell, we, do]",[],[]
1,"[professional, knowledgable, health, needs, co...","[professional, knowledgable, health, need, com...","[care, excellent]","[care, excellent]"
2,"[efficiency, system, overall, time, hours, see...","[efficiency, system, overall, time, hour, see,...","[continue, great, work, great work]","[continue, great, work]"
3,"[good, meet, emergency, need, emergency need]","[good, meet, emergency, need]",[],[]
4,[],[],"[please, install, drinking, water, fountains, ...","[please, install, drink, water, fountain]"
5,"[triage, minutes]","[triage, minute]","[visit, lasted, hours, impossible, times, find...","[visit, last, hour, impossible, time, find, ch..."
6,"[superb, impressed, devotion, staff]","[superb, impressed, devotion, staff]","[info, discharge, advice, given, one, weak, spot]","[info, discharge, advice, give, one, weak, spot]"
7,[fast],[fast],"[fast, waited, long, time, chest, results, lon...","[fast, wait, long, time, chest, result]"
8,[place],[place],"[placed, room, sick, persons, find, place, sit...","[place, room, sick, person, find, sit, take, p..."
9,"[timely, appointments, timely appointments]","[timely, appointment]",[],[]
